# Day 4 Lab - Write it yourself: CNNs for grey and colour images

In the code breaks you *changed* numbers. Now you **write the code yourself** - step by step:
- **Part 1 - A CNN for clothes images** (about 35 min): shapes by hand, a CNN from a plan, train it, look inside
- **Part 2 - Colour images: a CNN for CIFAR-10** (about 30 min): 3 colour channels, train it, data augmentation
- **⭐ Bonus** (homework): the most useful sanity check of all - overfit one batch

What is not finished in class is homework. A **CPU runtime is enough**.

How the writing tasks work:
1. read the **plan** (a small table),
2. write your code below the numbered steps - where it says `pass  # YOUR CODE HERE`,
3. stuck? open **💡 Hint**,
4. run the **check cell**: ✅ = correct, ❌ = it tells you what is wrong.

You cannot break anything - if something looks strange: *Runtime → Restart session*, then run the setup cells again.

## Setup - run these two cells first
The first cell loads FashionMNIST (28 × 28 pixel images of 10 kinds of clothes): 10 000 training,
2 000 validation and 2 000 test images - the same images as in Lab 3. **New:** the images stay images.
Each one has the shape `(1, 28, 28)` = (channels, height, width) - one channel, because they are grey.

In [ ]:
import copy

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

data = torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
X = data.data.float().unsqueeze(1) / 255            # 60 000 images, each (1, 28, 28)
y = data.targets

X_train, y_train = X[:10000], y[:10000]             # training images
X_val, y_val = X[50000:52000], y[50000:52000]       # validation images: for choosing
X_test, y_test = X[52000:54000], y[52000:54000]     # test images: only at the very end
mean, std = X_train.mean(), X_train.std()           # computed on the training images only
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std
print("training:", tuple(X_train.shape), "  validation:", tuple(X_val.shape), "  test:", tuple(X_test.shape))

The second cell defines the **check cells** and other helpers for the whole lab.
**Just run it - you do not need to read it.**

In [ ]:
from torchvision.transforms import v2

NAMES = ["T-shirt", "trouser", "pullover", "dress", "coat", "sandal", "shirt", "sneaker", "bag", "ankle boot"]


def _describe(layer):
    """('Conv2d', in, out, kernel, padding), ('MaxPool2d', k), ('Linear', in, out), ('ReLU',) ..."""
    name = type(layer).__name__
    if isinstance(layer, nn.Conv2d):
        return (name, layer.in_channels, layer.out_channels, layer.kernel_size[0], layer.padding[0])
    if isinstance(layer, nn.MaxPool2d):
        k = layer.kernel_size
        return (name, k if isinstance(k, int) else k[0])
    if isinstance(layer, nn.Linear):
        return (name, layer.in_features, layer.out_features)
    return (name,)


def _build(plan):
    """Build the reference network of a plan (only used by the checks)."""
    make = {"Conv2d": lambda a: nn.Conv2d(a[0], a[1], a[2], padding=a[3]), "MaxPool2d": lambda a: nn.MaxPool2d(a[0]),
            "Linear": lambda a: nn.Linear(a[0], a[1]), "ReLU": lambda a: nn.ReLU(), "Flatten": lambda a: nn.Flatten()}
    return nn.Sequential(*[make[p[0]](p[1:]) for p in plan])


PLAN_CNN = [("Conv2d", 1, 16, 3, 1), ("ReLU",), ("MaxPool2d", 2), ("Conv2d", 16, 32, 3, 1), ("ReLU",),
            ("MaxPool2d", 2), ("Flatten",), ("Linear", 32 * 7 * 7, 10)]
PLAN_CIFAR = [("Conv2d", 3, 16, 3, 1), ("ReLU",), ("MaxPool2d", 2), ("Conv2d", 16, 32, 3, 1), ("ReLU",),
              ("MaxPool2d", 2), ("Conv2d", 32, 64, 3, 1), ("ReLU",), ("MaxPool2d", 2), ("Flatten",),
              ("Linear", 64 * 4 * 4, 10)]


def _real_shapes(plan, x):
    shapes = []
    with torch.no_grad():
        for layer in _build(plan):
            x = layer(x)
            shapes.append(tuple(x.shape))
    return shapes


def _check_shapes(my_shapes, plan, x):
    real = _real_shapes(plan, x)
    if len(my_shapes) != len(real):
        print("❌ the plan has", len(real), "layers - your list has", len(my_shapes), "entries")
        return
    n_ok = 0
    for i, (p, mine, true) in enumerate(zip(plan, my_shapes, real)):
        if mine is None:
            print("❌", i + 1, p[0].ljust(18), "- not filled in yet")
        elif tuple(mine) != true:
            print("❌", i + 1, p[0].ljust(18), "yours:", tuple(mine), "  real:", true)
        else:
            n_ok += 1
            print("✅", i + 1, p[0].ljust(18), true)
    print(n_ok, "of", len(real), "shapes correct")


def check_shapes_cnn(my_shapes):
    _check_shapes(my_shapes, PLAN_CNN, torch.zeros(8, 1, 28, 28))


def check_shapes_cifar(my_shapes):
    _check_shapes(my_shapes, PLAN_CIFAR, torch.zeros(8, 3, 32, 32))


def _compare(model, plan, x, out_shape):
    """Compare a model with the plan; print ✅ or the first difference."""
    if model is None:
        print("❌ no model yet - write it in the cell above and run it again")
        return False
    if not isinstance(model, nn.Sequential):
        print("❌ please build the model with nn.Sequential(...)")
        return False
    mine = [_describe(layer) for layer in model]
    if len(mine) != len(plan):
        print("❌ the plan has", len(plan), "layers, your model has", len(mine), "-", [m[0] for m in mine])
        return False
    for i, (m, p) in enumerate(zip(mine, plan)):
        if m[0] != p[0]:
            print("❌ layer", i + 1, "should be", p[0], "- yours is", m[0])
            return False
        if m != p:
            print("❌ layer", i + 1, "(" + p[0] + ") should be", p[1:], "- yours is", m[1:])
            return False
    count = sum(p.numel() for p in model.parameters())
    n_plan = sum(p.numel() for p in _build(plan).parameters())
    if count != n_plan:
        print("❌ the model should have", n_plan, "parameters, yours has", count)
        return False
    with torch.no_grad():
        shape = tuple(model(x).shape)
    if shape != out_shape:
        print("❌ the output should have the shape", out_shape, "- yours has", shape)
        return False
    print("✅ the model matches the plan -", count, "parameters, output shape", shape)
    return True


def check_cnn(model):
    _compare(model, PLAN_CNN, torch.zeros(8, 1, 28, 28), (8, 10))


def check_cifar_cnn(make_cifar_cnn):
    _compare(make_cifar_cnn(), PLAN_CIFAR, torch.zeros(8, 3, 32, 32), (8, 10))


CROP_FLIP = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip()])   # augmentation for 2.4


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return round((model(X).argmax(1) == y).float().mean().item(), 3)


def train_model(model, lr=0.001, epochs=3, log_name=None, data=None, augment=False):
    """Your train_model from Lab 3 (given): train on the training images, return the validation accuracy.
    With log_name="..." the training is also written to TensorBoard (folder runs/log_name).
    data=(X_tr, y_tr, X_va, y_va) trains on other images (Part 2); augment=True: random crop + flip (Part 2)."""
    X_tr, y_tr, X_va, y_va = data if data is not None else (X_train, y_train, X_val, y_val)
    torch.manual_seed(0)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    writer = SummaryWriter(f"runs/{log_name}") if log_name else None
    step = 0
    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            if augment:
                x = torch.stack([CROP_FLIP(img) for img in x])           # a new random version of every image
            loss = loss_fn(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if writer:
                writer.add_scalar("loss/train", loss.item(), step)          # one point per batch
        print("epoch", epoch + 1, "  last batch loss:", round(loss.item(), 3))
        if writer:
            writer.add_scalar("accuracy/validation", accuracy(model, X_va, y_va), epoch + 1)    # one point per epoch
            writer.add_scalar("accuracy/train", accuracy(model, X_tr[:2000], y_tr[:2000]), epoch + 1)
    if writer:
        writer.close()
    return accuracy(model, X_va, y_va)


def make_mlp():
    """The MLP of Lab 3 (given) - with nn.Flatten() first, because the images are (1, 28, 28) here."""
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Dropout(p=0.2),
                         nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))


def show_filters_and_maps(model, image_nr=0):
    """Top: the 16 learned 3x3 filters of the first layer. Bottom: what each filter makes of one image."""
    conv = model[0]
    with torch.no_grad():
        maps = conv(X_val[image_nr:image_nr + 1])[0]
    fig, axes = plt.subplots(3, 16, figsize=(16, 3.4))
    for i in range(16):
        axes[0, i].imshow(conv.weight[i, 0].detach(), cmap="RdBu_r")
        axes[0, i].set_title("filter " + str(i), fontsize=8)
        axes[1, i].imshow(maps[i], cmap="gray")
        axes[2, i].axis("off")
    axes[2, 0].imshow(X_val[image_nr, 0], cmap="gray")
    axes[2, 0].set_title("input: " + NAMES[int(y_val[image_nr])], fontsize=8)
    for ax in axes[:2].ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def check_overfit(losses):
    if len(losses) == 0:
        print("❌ no losses yet - fill in the loop")
    elif losses[-1] > 0.05:
        print("❌ final loss", round(losses[-1], 3), "- should go down to almost 0. Did you call opt.step()?")
    else:
        print("✅ loss", round(losses[0], 2), "→", round(losses[-1], 4), "- the model CAN learn: no bug in the pipeline")


print("helpers ready")

# Part 1 - A CNN for clothes images (about 35 min)
## 1.1 Shapes by hand
Here is the plan of our CNN. **Before you build it**, write down the shape of the output after every layer for a
batch of **8 images** of shape `(8, 1, 28, 28)`. The first one is filled in.

Rules from the slides: `padding=1` with a 3 × 3 kernel keeps height and width; `MaxPool2d(2)` halves them;
a conv layer sets the number of channels to its `out` number; `Flatten` multiplies everything except the batch.

| # | layer | in | out | kernel | note |
|---|---|---|---|---|---|
| 1 | `nn.Conv2d` | 1 | 16 | 3 | `padding=1` |
| 2 | `nn.ReLU` | | | | |
| 3 | `nn.MaxPool2d` | | | 2 | |
| 4 | `nn.Conv2d` | 16 | 32 | 3 | `padding=1` |
| 5 | `nn.ReLU` | | | | |
| 6 | `nn.MaxPool2d` | | | 2 | |
| 7 | `nn.Flatten` | | | | |
| 8 | `nn.Linear` | 32 · 7 · 7 = 1568 | 10 | | one raw score per kind of clothes |

In [ ]:
my_shapes = [
    (8, 16, 28, 28),   # 1 Conv2d  (example - already filled in)
    None,              # 2 ReLU
    None,              # 3 MaxPool2d
    None,              # 4 Conv2d
    None,              # 5 ReLU
    None,              # 6 MaxPool2d
    None,              # 7 Flatten
    None,              # 8 Linear
]
# 1. replace every None by a shape like (8, 16, 28, 28)
pass  # YOUR CODE HERE
check_shapes_cnn(my_shapes)

<details><summary>💡 Hint</summary>

- ReLU never changes the shape
- `MaxPool2d(2)`: 28 → 14, 14 → 7 - the channels stay
- `Flatten`: (8, 32, 7, 7) → (8, 32 · 7 · 7)
- the last layer gives one number per class: (8, 10)
</details>

## 1.2 The CNN from the plan
Build the network from the table in 1.1 with `nn.Sequential(...)`.

In [ ]:
torch.manual_seed(0)    # same start weights for everybody
cnn = None
# 1. cnn = nn.Sequential( ... the 8 layers from the plan, separated by commas ... )
pass  # YOUR CODE HERE
print(cnn)

<details><summary>💡 Hint</summary>

- a conv layer with 1 input channel, 16 filters of 3 × 3 that keeps the size: `nn.Conv2d(1, 16, kernel_size=3, padding=1)`
- pooling: `nn.MaxPool2d(2)` - flatten: `nn.Flatten()` - all with `()`
- the last layer: `nn.Linear(32 * 7 * 7, 10)`
</details>

In [ ]:
check_cnn(cnn)

❓ The MLP of Lab 3 had **235 146** parameters. How many does the CNN have - and where are most of them?

*Your answer:*

<details><summary>👀 What you should see</summary>

- ✅ **20 490** parameters - more than 10× fewer than the MLP
- conv 1: 16 · (1 · 3 · 3) + 16 = **160**, conv 2: 32 · (16 · 3 · 3) + 32 = **4 640**,
  linear: 1568 · 10 + 10 = **15 690** - most are in the last (fully connected) layer
- the two conv layers have only 4 800 parameters: a 3 × 3 filter is **shared** over the whole image
</details>

## 1.3 Train it
`train_model(model, lr, epochs)` is your function from Lab 3 (given in the helper cell). Train the CNN and
compare with the MLP of Lab 3: **0.836** validation accuracy after 3 epochs.

In [ ]:
LR = 0.001      # 👈
EPOCHS = 3      # 👈

if cnn is not None:
    trained_cnn = copy.deepcopy(cnn)    # a copy of your untrained CNN - so every run starts the same
    print("CNN validation accuracy:", train_model(trained_cnn, lr=LR, epochs=EPOCHS), "  (MLP of Lab 3: 0.836)")

❓ **Explain in one sentence:** why is the CNN better than the MLP - with more than 10× fewer parameters?

*Your answer:*

<details><summary>👀 What you should see</summary>

- last batch loss **0.47 → 0.41 → 0.23**, CNN validation accuracy **0.855** (MLP: 0.836)
- about 20 s on a CPU (the MLP took a few seconds - the convolutions cost compute, not parameters)
- answer: the CNN keeps the image as an image - its small filters look at **neighbouring pixels** and the **same
  filter** is used everywhere, so a pattern (an edge, a collar) is found wherever it is. The MLP sees 784
  unrelated numbers and has to learn every position separately.
</details>

## 1.4 Look inside
Top row: the 16 learned 3 × 3 filters of the first layer (red = positive, blue = negative weights).
Middle row: what each filter makes of one validation image. Bottom left: the input image.

In [ ]:
IMAGE_NR = 0    # 👈 try other images: 1, 2, 3, ...

if cnn is not None:
    show_filters_and_maps(trained_cnn, IMAGE_NR)

❓ Which filters look like the hand-made **edge kernels** from the slides (one side positive, the other negative)?
What do their feature maps show?

*Your answer:*

<details><summary>👀 What you should see</summary>

- several filters have a red half and a blue half (left/right or top/bottom) - **edge detectors**, like the
  hand-made kernels; their feature maps are bright only along the outlines of the clothes
- other filters are almost all red or all blue - they react to bright or dark areas
- nobody programmed these filters: the CNN **learned** them with backpropagation in 3 epochs
</details>

## 1.5 Watch the training in TensorBoard   ⏱ 10 min
**Goal:** see the training of the CNN and of the MLP from Lab 3 as curves - side by side, in one picture.

**Your job**
1. ▶ Run the next cell. It trains the MLP and your CNN again and writes every step to TensorBoard
   (folder `runs/`). This takes about 1 minute.
2. ▶ Run the TensorBoard cell below it. TensorBoard opens inside the notebook.
3. ❓ Compare the two runs and answer the question at the end.

In [ ]:
EPOCHS_TB = 5      # 👈

if cnn is not None:
    print("MLP:", train_model(make_mlp(), epochs=EPOCHS_TB, log_name="mlp"))
    print("CNN:", train_model(copy.deepcopy(cnn), epochs=EPOCHS_TB, log_name="cnn"))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In TensorBoard: tab **SCALARS** (or **TIME SERIES**). `loss/train` has one point per batch, `accuracy/validation`
one point per epoch. Each run (`mlp`, `cnn`) is one coloured line - you can switch runs on and off on the left.
Move the **smoothing** slider to about 0.8 to see the trend of the noisy loss.

❓ **Explain in one sentence:** which model learns faster, and which one is better on the validation images?

*Your answer:*

<details><summary>👀 What you should see</summary>

- after 5 epochs: MLP validation accuracy **0.853**, CNN **0.865** (about 1 minute on a CPU)
- `accuracy/validation`: the two lines are close and can cross in between; after 5 epochs the CNN is ahead.
  `loss/train` is very noisy (one point per batch of 64 images) - move the smoothing slider to about 0.8
- answer: after a few epochs the CNN is a bit better on the validation images - with 10x fewer weights, because its
  filters use the neighbourhood of the pixels (task 1.3).
- the same tool works for every PyTorch training: `SummaryWriter("runs/name")`, `writer.add_scalar(...)`
  (slide "Watch your training: TensorBoard")
</details>

# Part 2 - Colour images: a CNN for CIFAR-10 (about 30 min)
CIFAR-10: 32 × 32 **colour** photos of 10 kinds of objects - planes, cars, birds, cats, deer, dogs, frogs, horses,
ships, trucks. New: every image has **3 channels** (red, green, blue), so its shape is `(3, 32, 32)`.
We use 10 000 training and 2 000 validation images, so that a CPU is enough. The download (170 MB) takes about 1 minute.

In [ ]:
cifar = torchvision.datasets.CIFAR10(root="data", train=True, download=True)
CIFAR_NAMES = cifar.classes
Xc = torch.tensor(cifar.data).permute(0, 3, 1, 2).float() / 255     # (50 000, 3, 32, 32): channels first!
yc = torch.tensor(cifar.targets)

Xc_train, yc_train = Xc[:10000], yc[:10000]           # training images
Xc_val, yc_val = Xc[40000:42000], yc[40000:42000]     # validation images
c_mean = Xc_train.mean(dim=(0, 2, 3), keepdim=True)   # one mean per colour channel - training images only
c_std = Xc_train.std(dim=(0, 2, 3), keepdim=True)
Xc_train = (Xc_train - c_mean) / c_std
Xc_val = (Xc_val - c_mean) / c_std
CIFAR = (Xc_train, yc_train, Xc_val, yc_val)          # everything train_model needs
print("training:", tuple(Xc_train.shape), "  validation:", tuple(Xc_val.shape))

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for i in range(10):
    axes[i].imshow(cifar.data[i])
    axes[i].set_title(CIFAR_NAMES[cifar.targets[i]], fontsize=9)
    axes[i].axis("off")
plt.show()

## 2.1 Shapes by hand
The plan of the CIFAR CNN: three conv blocks instead of two. Write down the output shape after every layer for
a batch of **8 images** of shape `(8, 3, 32, 32)`. The first one is filled in.

| # | layer | in | out | kernel | note |
|---|---|---|---|---|---|
| 1 | `nn.Conv2d` | 3 | 16 | 3 | `padding=1` - 3 input channels: red, green, blue |
| 2 | `nn.ReLU` | | | | |
| 3 | `nn.MaxPool2d` | | | 2 | |
| 4 | `nn.Conv2d` | 16 | 32 | 3 | `padding=1` |
| 5 | `nn.ReLU` | | | | |
| 6 | `nn.MaxPool2d` | | | 2 | |
| 7 | `nn.Conv2d` | 32 | 64 | 3 | `padding=1` |
| 8 | `nn.ReLU` | | | | |
| 9 | `nn.MaxPool2d` | | | 2 | |
| 10 | `nn.Flatten` | | | | |
| 11 | `nn.Linear` | 64 · 4 · 4 = 1024 | 10 | | one raw score per kind of object |

In [ ]:
my_cifar_shapes = [
    (8, 16, 32, 32),   # 1 Conv2d  (example - already filled in)
    None,              # 2 ReLU
    None,              # 3 MaxPool2d
    None,              # 4 Conv2d
    None,              # 5 ReLU
    None,              # 6 MaxPool2d
    None,              # 7 Conv2d
    None,              # 8 ReLU
    None,              # 9 MaxPool2d
    None,              # 10 Flatten
    None,              # 11 Linear
]
# 1. replace every None by a shape like (8, 16, 32, 32)
pass  # YOUR CODE HERE
check_shapes_cifar(my_cifar_shapes)

<details><summary>💡 Hint</summary>

- the same rules as in 1.1 - the image size goes 32 → 16 → 8 → 4 (three times `MaxPool2d(2)`)
- the channels are set by the conv layers: 16, 32, 64
- `Flatten`: (8, 64, 4, 4) → (8, 64 · 4 · 4)
</details>

## 2.2 The CIFAR CNN from the plan
Write `make_cifar_cnn()`: the 11 layers of the plan in `nn.Sequential(...)`. It is a function, so that every
experiment can start with a fresh, untrained CNN.

In [ ]:
def make_cifar_cnn():
    torch.manual_seed(0)    # same start weights for everybody
    model = None
    # 1. model = nn.Sequential( ... the 11 layers from the plan ... )
    pass  # YOUR CODE HERE
    return model


check_cifar_cnn(make_cifar_cnn)

<details><summary>💡 Hint</summary>

- the same as 1.2, but the first layer gets **3** input channels: `nn.Conv2d(3, 16, kernel_size=3, padding=1)`
- one more block: `nn.Conv2d(32, 64, kernel_size=3, padding=1)`, `nn.ReLU()`, `nn.MaxPool2d(2)`
- the last layer: `nn.Linear(64 * 4 * 4, 10)`
</details>

❓ How many parameters does the CIFAR CNN have? Why does the **first** conv layer have more weights than in 1.2?

*Your answer:*

<details><summary>👀 What you should see</summary>

- ✅ **33 834** parameters, output shape **(8, 10)**
- first layer: 16 · (**3** · 3 · 3) + 16 = **448** instead of 160 - every filter looks at all 3 colour channels
</details>

## 2.3 Train it
`train_model` (given) trains the CNN on the CIFAR images for 15 epochs (about 1 minute) and writes the run to
TensorBoard (`runs/cifar`). It prints the validation accuracy; the cell also prints the accuracy on 2 000
**training** images.

In [ ]:
EPOCHS_C = 15     # 👈 about 1 minute per run on a CPU

cifar_cnn = make_cifar_cnn()
if cifar_cnn is not None:
    print("validation accuracy:", train_model(cifar_cnn, epochs=EPOCHS_C, data=CIFAR, log_name="cifar"))
    print("training accuracy:  ", accuracy(cifar_cnn, Xc_train[:2000], yc_train[:2000]))

❓ **Explain in one sentence:** why is CIFAR-10 so much harder than FashionMNIST - and what does the gap between
training and validation accuracy tell you?

*Your answer:*

<details><summary>👀 What you should see</summary>

- validation accuracy **0.60**, training accuracy **0.85** after 15 epochs (about 1 minute on a CPU) -
  FashionMNIST reached 0.86 after only 3 epochs
- in TensorBoard (`cifar`): `accuracy/train` keeps climbing, `accuracy/validation` flattens out at about 0.6
- answer: the objects have colours, different backgrounds, sizes and poses (a cat from the front, from the side,
  half hidden), while FashionMNIST shows centred clothes on black. The gap between training and validation
  accuracy means the CNN starts to **learn the training images by heart** - overfitting (day 3).
</details>

## 2.4 Data augmentation against overfitting
`augment=True` gives the CNN a **new random version** of every training image in every epoch: shifted a little
(random crop with padding 4) and maybe mirrored left-right - a car is still a car (day 3, block 2).

🔧 Run the cell, then compare `cifar` and `cifar_aug` in TensorBoard (the TensorBoard cell of 1.5 - click ⟳ to
refresh, or run it again). Look at `accuracy/train` and `accuracy/validation`.

In [ ]:
AUGMENT = True     # 👈

cifar_cnn_aug = make_cifar_cnn()
if cifar_cnn_aug is not None:
    print("validation accuracy:", train_model(cifar_cnn_aug, epochs=EPOCHS_C, data=CIFAR, augment=AUGMENT,
                                              log_name="cifar_aug"))
    print("training accuracy:  ", accuracy(cifar_cnn_aug, Xc_train[:2000], yc_train[:2000]))

❓ **Explain in one sentence:** what happens to the gap between training and validation accuracy with augmentation?

*Your answer:*

<details><summary>👀 What you should see</summary>

- with augmentation: validation accuracy **0.64**, training accuracy **0.69** - the gap shrinks from **0.25** to
  **0.05**, and the validation accuracy is *higher* than without augmentation (0.60)
- in TensorBoard: the `cifar_aug` training curve rises more slowly (the images are harder every epoch), but its
  validation curve ends above the one of `cifar`
- answer: the CNN can no longer learn the exact training images by heart, because it never sees the same version
  twice - less overfitting, better on new images. Training takes a bit longer (about 1.3 minutes).
</details>

## ⭐ Bonus: overfit one batch (homework)
Before a long training run, professionals do one quick test: can the model learn **one single batch** by heart?
If the loss does not go to almost 0, there is a bug in the pipeline (wrong labels, missing `opt.step()`, ...).

✏️ **Write the loop:** 200 steps on the same batch `xb, yb`, and append every loss to `losses`.

In [ ]:
losses = []
if cnn is not None:
    xb, yb = X_train[:64], y_train[:64]         # ONE batch of 64 images
    model_b = copy.deepcopy(cnn)                # a fresh copy of your CNN
    opt = torch.optim.Adam(model_b.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    model_b.train()
    # 1. for step in range(200):
    # 2.     loss = loss_fn(model_b(xb), yb)
    # 3.     zero_grad, backward, step
    # 4.     losses.append(loss.item())
    pass  # YOUR CODE HERE
check_overfit(losses)

<details><summary>👀 What you should see</summary>

- ✅ loss **2.3 → 0.0013** - the CNN learns the 64 images by heart: the pipeline works
- 2.3 at the start = ln(10): a fresh model guesses all 10 classes equally - another quick sanity check
</details>

# For your project
Before you train for hours, check:
- **shapes**: print the output shape after every layer for a batch of 2 (or compute them by hand, as in 1.1)
- **parameters**: `sum(p.numel() for p in model.parameters())` - where are most of them?
- **overfit one batch**: the loss must go to almost 0 - otherwise there is a bug
- **first loss** ≈ ln(number of classes) for a fresh classifier with balanced classes
- **normalise** with the mean and spread of the **training** images only (day 3)
- **track every run** (TensorBoard as in 1.5; for the challenge W&B, MLflow or ClearML) - and change one thing at a time
- **overfitting?** compare training and validation accuracy; augmentation, dropout, weight decay help (day 3)
- **imbalanced classes** (like in the challenge): balanced accuracy and sensitivity per class - never accuracy alone